<a href="https://colab.research.google.com/github/madanjha/PythonDS/blob/main/splunklogdwnld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install splunk-sdk streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.2/109.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.2 MB/s eta 0:00:00
  Created wheel for splunk-sdk: filename=splunk_sdk-2.1.1-py3-none-any.whl size=125851 sha256=223127130fca0ec486cae7b89f05ec7ed22176d8e1a83311110fae1352e83aad
  Stored in directory: /root/.cache/pip/wheels/03/4c/9b/b766a39da99db682a378fdd526d598d975020b2a0e0d9a2421
Successfully built splunk-sdk


In [2]:
import streamlit as st
import splunklib.client as client
import os
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def generate_filename(filename1, second_param, namespace):
    """Generates the filename based on the specified naming convention."""
    return f"{filename1}_{second_param}_{namespace}_{second_param}.log"

def splunk_connect(host, port, username, password_or_token):
    """Establishes a connection to the Splunk server."""
    try:
        service = client.connect(
            host=host,
            port=port,
            username=username,
            password=password_or_token,
            autologin=True
        )
        return service
    except Exception as e:
        logging.error(f"Connection failed: {e}")
        raise e

def execute_splunk_query(service, query):
    """Executes a search query and returns the job."""
    try:
        job = service.jobs.export(query, output_mode="csv")
        return job
    except Exception as e:
        logging.error(f"Query execution failed: {e}")
        raise e

def main():
    st.title("Splunk Log Downloader")

    with st.sidebar:
        st.header("Connection Settings")
        host = st.text_input("Splunk Host", value="localhost")
        port = st.number_input("Port", value=8089)
        username = st.text_input("Username")
        secret = st.text_input("Password or Token", type="password")

    st.header("Query Parameters")
    query = st.text_area("Splunk Search Query", value="search index=_internal | head 10")
    filename1 = st.text_input("Base Filename (filename1)", value="applicationlogs")
    namespace = st.text_input("Namespace Parameter")
    second_param = st.text_input("Second Parameter (e.g., prod)")
    download_path = st.text_input("Download Directory", value="./")

    if st.button("Fetch and Download Logs"):
        if not all([host, username, secret, query, namespace, second_param]):
            st.error("Please fill in all fields.")
            return

        try:
            st.info("Connecting to Splunk...")
            service = splunk_connect(host, port, username, secret)

            st.info("Executing query...")
            results_stream = execute_splunk_query(service, query)

            target_filename = generate_filename(filename1, second_param, namespace)
            full_path = os.path.join(download_path, target_filename)

            st.info("Streaming results to file...")
            with open(full_path, 'wb') as f:
                for chunk in results_stream:
                    # Simple filtering logic based on namespace if found in string
                    if namespace.encode() in chunk:
                         f.write(chunk)
                    elif not namespace:
                         f.write(chunk)

            st.success(f"Logs downloaded successfully to: {full_path}")
            logging.info(f"Successfully saved logs to {full_path}")

        except Exception as e:
            st.error(f"An error occurred: {e}")

if __name__ == '__main__':
    # Note: To run this in Colab, you would normally save this to a file
    # and run 'streamlit run app.py' with a tunnel like ngrok.
    # For this demonstration, we define the logic here.
    main()

2026-05-22 04:59:23.079 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.286 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-22 04:59:23.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.288 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.291 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 04:59:23.292 Thread 'MainThread': mi

In [3]:
%%writefile app.py
import streamlit as st
import splunklib.client as client
import os
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def generate_filename(filename1, second_param, namespace):
    """Generates the filename based on the specified naming convention."""
    return f"{filename1}_{second_param}_{namespace}_{second_param}.log"

def splunk_connect(host, port, username, password_or_token):
    """Establishes a connection to the Splunk server using splunk-sdk."""
    try:
        service = client.connect(
            host=host,
            port=port,
            username=username,
            password=password_or_token,
            autologin=True
        )
        return service
    except Exception as e:
        logging.error(f"Connection failed: {e}")
        raise e

def execute_splunk_query(service, query):
    """Executes a search query and returns an export job stream."""
    try:
        # Using export for large datasets
        job = service.jobs.export(query, output_mode="csv")
        return job
    except Exception as e:
        logging.error(f"Query execution failed: {e}")
        raise e

def main():
    st.set_page_config(page_title="Splunk Log Fetcher", layout="wide")
    st.title("Splunk Log Downloader Application")

    # Input UI
    with st.sidebar:
        st.header("1. Connection Details")
        host = st.text_input("Splunk Host URL", value="localhost")
        port = st.number_input("Management Port", value=8089)
        user = st.text_input("Username")
        pw = st.text_input("Password / Token", type="password")

    col1, col2 = st.columns(2)
    with col1:
        st.header("2. Search & Parameters")
        query = st.text_area("Splunk Search Query", value="search index=_internal | head 100")
        ns = st.text_input("Namespace (Filter)", help="Filter logs containing this string")
        sp = st.text_input("Second Parameter (e.g. env)", value="prod")

    with col2:
        st.header("3. Output Settings")
        fn1 = st.text_input("Base Filename (filename1)", value="applicationlogs")
        dl_path = st.text_input("Local Save Path", value="./")

    if st.button("Run Job and Download"):
        if not all([host, user, pw, query, ns, sp]):
            st.warning("All input fields are required.")
            return

        try:
            # Connection
            service = splunk_connect(host, port, user, pw)
            st.success("Connected to Splunk!")

            # Filename Generation
            target_name = generate_filename(fn1, sp, ns)
            full_out_path = os.path.join(dl_path, target_name)

            # Execution & Stream handling
            with st.spinner("Fetching logs..."):
                results = execute_splunk_query(service, query)
                count = 0
                with open(full_out_path, 'wb') as f:
                    for chunk in results:
                        # Applying namespace filter if provided
                        if ns.encode() in chunk:
                            f.write(chunk)
                            count += 1

            st.success(f"Finished! Saved to {full_out_path}")
            st.info(f"Filtered chunks written: {count}")
            logging.info(f"Job complete. File: {target_name}")

        except Exception as e:
            st.error(f"Application Error: {str(e)}")

if __name__ == '__main__':
    main()

Writing app.py


In [4]:
import urllib
print("Password for localtunnel (IP address):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
!streamlit run app.py & npx localtunnel --port 8501

Password for localtunnel (IP address): 34.6.254.201


⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-05-22 04:59:47.801 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.6.254.201:8501

  Stopping...
^C


### How to run the UI in Google Colab
Streamlit requires a local server. To view the UI in Colab:
1. Save the code above into a file named `app.py`.
2. Run `!streamlit run app.py & npx localtunnel --port 8501` in a new cell.
3. Click the link provided by localtunnel to open the UI.